# SOC_exemple_Github

This notebook is a compact example of how to use `SolverV9` to calculate a
pathway-resolved 2D spectrum for a dynamical model. The physical model is kept
from `SOC_workbench_essaie`; the comments focus on the solver workflow rather
than the spin/phonon model details.

The key `SolverV9` workflow is:

1. build a Hamiltonian, an interaction operator, optional collapse operators, and an initial density matrix;
2. feed them to `LiouvilleSpectroscopySolver`;
3. generate explicit UFSS pathways with `generate_pathways_with_ufss(...)`;
4. define the spectroscopy protocol with `standard_nq_protocol(...)`;
5. compute spectra with `generate_NQ_spectrum(...)` using `delays={...}`;
6. optionally repeat the calculation in a loop over a parameter dictionary.


## Imports and project setup

This cell locates the repository root, imports `SolverV9`, and defines output
folders. For a standalone project, the only required solver imports are:

```python
from SolverV9 import LiouvilleSpectroscopySolver, SpectroscopyPlotter, standard_nq_protocol
```

`SolverV9` does not assume default rephasing/unrephasing pathways. Pathways must
be generated or supplied explicitly.


In [1]:
from pathlib import Path
from datetime import datetime
import csv
import json
import sys

import matplotlib

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "SolverV9").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate PROJECT_ROOT containing SolverV9.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from SolverV9 import LiouvilleSpectroscopySolver, SpectroscopyPlotter, standard_nq_protocol

RESULT_ROOT = PROJECT_ROOT / "SOC Model" / "result_3_0"
CODE_DIR = RESULT_ROOT / "code"
DATA_DIR = RESULT_ROOT / "Data"
FIGURES_DIR = RESULT_ROOT / "Figures"
ANALYSIS_DIR = RESULT_ROOT / "Analysis"
REPORT_DIR = RESULT_ROOT / "report"
SOURCE_NOTEBOOK = (
    PROJECT_ROOT
    / "SOC Model"
    / "Result_Test"
    / "etape_3"
    / "step3_dynamic_dimerisation_phonon.ipynb"
)

for directory in (CODE_DIR, DATA_DIR, FIGURES_DIR, ANALYSIS_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)



## Parameters and scan definitions



## User parameters

The workbench is controlled by two dictionaries.

`model_params` describes the physical model. In your own model, this dictionary
can contain any parameters needed to build `H`, the light-matter interaction
operator, dissipation operators, and the initial density matrix.

`solver_params` controls numerical broadening, backend choice, and parallel
execution. For `SolverV9`, `parallel_backend="threading"` and `n_jobs=-1`
parallelize the general prefix-tree spectrum calculation over frequency-axis
blocks.

`spectrum_params` controls the frequency grid and the fixed waiting time. In
`SolverV9`, fixed times are passed later through the `delays` dictionary, not
through a special `tau2` argument.


In [2]:
K_B_EV_K = 8.617333262145e-5
MU_B_EV_T = 5.7883818060e-5
matched_lambda_C = 0.01 / -0.443

model_params = {
    "Delta_dark": 0.90,
    "Delta_Bright": 1.00,
    "mu_B": 1.0,
    "mu_D": 0.0,
    "V0": 0.1,
    "lambda_delta": 0.20,
    "lambda_C": matched_lambda_C,
    "delta_source": "spin_peierls_mean_field",
    "delta_external": 0.0,
    "C1": 0.0,
    "spin_correlation_source": "exact_spin_chain",
    "n_bosons": 3,
    "omega_Q": 0.035,
    "g_Q": 0.010,
    "phonon_displacement_couples_to": "bright_dark",
    "T": 7.0,
    "T_SP_0": 14.0,
    "B": 0.0,
    "delta_0": 0.01,
    "beta": 0.5,
    "alpha_field": 0.004,
    "N_spin": 6,
    "J_spin_eV": 0.010,
    "J2_spin_eV": 0.0,
    "g_factor": 2.0,
    "boundary": "open",
    "N_k": 1,
    "gamma_orb": 0.0,
    "gamma_phonon": 0.0,
}

output_stem = "SOC_workbench"

solver_params = {
    "T": 0.0,
    "Eta": 0.005,
    "backend": "dense",
    "parallel_backend": "threading",
    "n_jobs": -1,
}

spectrum_params = {
    "N_w": 150,
    "omega1_rephasing_min": -1.35,
    "omega1_rephasing_max": -0.65,
    "omega1_unrephasing_min": 0.65,
    "omega1_unrephasing_max": 1.35,
    "omega3_min": 0.65,
    "omega3_max": 1.35,
    "tau2": 2000.0,
}



## Explicit spin-chain model



## Model construction: what the solver needs

The details below are specific to this SOC example, but the solver only needs a
few standard objects:

- `H_model`: Hamiltonian in the site/model basis, shape `(d, d)` or `(N_k, d, d)`;
- `interaction_op_array`: optical/current/dipole interaction operator, same shape convention;
- `c_ops_raw`: optional Lindblad collapse operators, each as `C` or `(C, gamma)`;
- `initial_density_matrix`: optional initial density matrix, same shape convention;
- optional `k_array` and `k_weights` if the model has momentum samples.

Practical checks before calling `feed_model(...)`: `H` should be Hermitian, the
interaction operator should connect the states addressed by the optical
sequence, and `rho0` should be normalized and compatible with the basis
specified by `density_matrix_basis`.


In [3]:
def spin_peierls_transition_temperature(B, T_SP_0, alpha_field):
    return max(0.0, float(T_SP_0) * (1.0 - float(alpha_field) * float(B) ** 2))


def spin_peierls_delta(T, B, T_SP_0, delta_0, beta, alpha_field):
    T_SP = spin_peierls_transition_temperature(B, T_SP_0, alpha_field)
    if T_SP > 0.0 and float(T) < T_SP:
        return float(delta_0) * (1.0 - float(T) / T_SP) ** float(beta)
    return 0.0


def resolve_delta(params):
    if params["delta_source"] == "spin_peierls_mean_field":
        return spin_peierls_delta(
            params["T"],
            params["B"],
            params["T_SP_0"],
            params["delta_0"],
            params["beta"],
            params["alpha_field"],
        )
    return float(params["delta_external"])


def single_site_spin_matrices():
    sx = 0.5 * np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
    sy = 0.5 * np.array([[0.0, -1.0j], [1.0j, 0.0]], dtype=complex)
    sz = 0.5 * np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)
    return sx, sy, sz


def kron_all(operators):
    out = np.array([[1.0]], dtype=complex)
    for op in operators:
        out = np.kron(out, op)
    return out


def spin_site_operators(N_spin):
    sx, sy, sz = single_site_spin_matrices()
    ident = np.eye(2, dtype=complex)
    site_ops = []
    for site in range(N_spin):
        ops = []
        for alpha in (sx, sy, sz):
            factors = [ident] * N_spin
            factors[site] = alpha
            ops.append(kron_all(factors))
        site_ops.append(tuple(ops))
    return site_ops


def spin_dot_operator(site_ops, i, j):
    return sum(site_ops[i][axis] @ site_ops[j][axis] for axis in range(3))


def nearest_neighbor_pairs(N_spin, boundary):
    pairs = [(i, i + 1) for i in range(N_spin - 1)]
    if boundary == "periodic" and N_spin > 2:
        pairs.append((N_spin - 1, 0))
    return pairs


def next_nearest_neighbor_pairs(N_spin, boundary):
    pairs = [(i, i + 2) for i in range(N_spin - 2)]
    if boundary == "periodic" and N_spin > 3:
        pairs.extend([(N_spin - 2, 0), (N_spin - 1, 1)])
    return pairs


def build_spin_hamiltonian(params, delta):
    N_spin = int(params["N_spin"])
    boundary = str(params["boundary"])
    dim = 2**N_spin
    site_ops = spin_site_operators(N_spin)
    H = np.zeros((dim, dim), dtype=complex)
    bond_ops = []

    for i, j in nearest_neighbor_pairs(N_spin, boundary):
        bond = spin_dot_operator(site_ops, i, j)
        J_i = float(params["J_spin_eV"]) * (1.0 + ((-1.0) ** i) * float(delta))
        H += J_i * bond
        bond_ops.append(bond)

    for i, j in next_nearest_neighbor_pairs(N_spin, boundary):
        H += float(params["J2_spin_eV"]) * spin_dot_operator(site_ops, i, j)

    zeeman = sum(site_ops[i][2] for i in range(N_spin))
    H += float(params["g_factor"]) * MU_B_EV_T * float(params["B"]) * zeeman
    return H, bond_ops


def thermal_expectation_from_eigensystem(evals, evecs, operator, T):
    op_eigen_diag = np.einsum("ij,ij->j", evecs.conj(), operator @ evecs)
    if float(T) <= 0.0:
        e0 = float(np.min(evals))
        mask = np.isclose(evals, e0, atol=1e-12)
        return complex(np.mean(op_eigen_diag[mask]))
    beta = 1.0 / (K_B_EV_K * float(T))
    shifted = evals - float(np.min(evals))
    weights = np.exp(-beta * shifted)
    return complex(np.dot(weights, op_eigen_diag) / np.sum(weights))


def exact_spin_chain_observables(params, delta):
    H_spin, bond_ops = build_spin_hamiltonian(params, delta)
    hermitian_residual = float(np.max(np.abs(H_spin - H_spin.conj().T)))
    evals, evecs = np.linalg.eigh(H_spin)
    bond_values = [
        thermal_expectation_from_eigensystem(evals, evecs, op, params["T"]).real
        for op in bond_ops
    ]
    return {
        "C1": float(np.mean(bond_values)),
        "bond_C1_values": np.array(bond_values),
        "spin_dimension": int(H_spin.shape[0]),
        "spin_hilbert_dimension_expected": int(2 ** int(params["N_spin"])),
        "spin_hermitian_residual": hermitian_residual,
        "spin_ground_energy_eV": float(evals[0]),
        "spin_gap_eV": float(evals[1] - evals[0]) if len(evals) > 1 else 0.0,
    }



## Projected optical bright/dark/phonon model



## Build the projected optical model

This section maps the dictionary parameters to the actual matrices used by the
solver. The specific bright/dark/phonon construction is model-dependent. For a
different model, keep the same interface: return `H`, `mu`, optional `c_ops`,
`rho0`, and metadata.

The helper `make_solver(params)` is the important boundary: it builds the
matrices, creates `LiouvilleSpectroscopySolver`, and calls `feed_model(...)`.


In [4]:
def static_bright_dark_mixing(params, delta, C1):
    return (
        float(params["V0"])
        + float(params["lambda_delta"]) * float(delta)
        + float(params["lambda_C"]) * float(C1)
    )


def lowering_operator(n):
    op = np.zeros((n, n), dtype=complex)
    for upper in range(1, n):
        op[upper - 1, upper] = np.sqrt(upper)
    return op


def build_spin_resolved_projected_model(params):
    N_k = int(params["N_k"])
    if N_k != 1:
        raise NotImplementedError("result_3_0 keeps N_k=1 to isolate the spin-chain control.")
    k_array = np.array([0.0])
    k_weights = np.ones(1)

    delta = resolve_delta(params)
    spin_observables = exact_spin_chain_observables(params, delta)
    if params["spin_correlation_source"] == "exact_spin_chain":
        C1 = spin_observables["C1"]
    else:
        C1 = float(params["C1"])
    V_BD_static = static_bright_dark_mixing(params, delta, C1)

    n_bosons = int(params["n_bosons"])
    ket_g, ket_d, ket_b = 0, 1, 2
    H_orb = np.zeros((3, 3), dtype=complex)
    H_orb[ket_d, ket_d] = params["Delta_dark"]
    H_orb[ket_b, ket_b] = params["Delta_Bright"]

    L_bd = np.zeros((3, 3), dtype=complex)
    L_bd[ket_d, ket_b] = 1.0
    L_bd[ket_b, ket_d] = 1.0

    mu_orb = np.zeros((3, 3), dtype=complex)
    mu_orb[ket_g, ket_b] = params["mu_B"]
    mu_orb[ket_b, ket_g] = params["mu_B"]
    mu_orb[ket_g, ket_d] = params["mu_D"]
    mu_orb[ket_d, ket_g] = params["mu_D"]

    a = lowering_operator(n_bosons)
    adag = a.conj().T
    n_op = adag @ a
    q_op = a + adag

    I_orb = np.eye(3, dtype=complex)
    I_ph = np.eye(n_bosons, dtype=complex)
    dim = 3 * n_bosons

    H = (
        np.kron(H_orb, I_ph)
        + float(params["omega_Q"]) * np.kron(I_orb, n_op)
        + V_BD_static * np.kron(L_bd, I_ph)
        + float(params["g_Q"]) * np.kron(L_bd, q_op)
    )
    mu = np.kron(mu_orb, I_ph)
    rho0 = np.zeros((dim, dim), dtype=complex)
    rho0[0, 0] = 1.0

    c_ops = []
    if params["gamma_phonon"]:
        c_ops.append((np.kron(I_orb, a)[None, :, :], params["gamma_phonon"]))
    if params["gamma_orb"]:
        C_bg = np.zeros((3, 3), dtype=complex)
        C_bg[ket_g, ket_b] = 1.0
        C_dg = np.zeros((3, 3), dtype=complex)
        C_dg[ket_g, ket_d] = 1.0
        c_ops.append((np.kron(C_bg, I_ph)[None, :, :], params["gamma_orb"]))
        c_ops.append((np.kron(C_dg, I_ph)[None, :, :], params["gamma_orb"]))

    meta = {
        "k_array": k_array,
        "k_weights": k_weights,
        "T_SP": spin_peierls_transition_temperature(
            params["B"], params["T_SP_0"], params["alpha_field"]
        ),
        "delta": float(delta),
        "C1": float(C1),
        "V_BD_static": float(V_BD_static),
        "omega_Q": float(params["omega_Q"]),
        "g_Q": float(params["g_Q"]),
        "dim": dim,
        **spin_observables,
    }
    return H[None, :, :], mu[None, :, :], c_ops, rho0[None, :, :], meta


def make_solver(params):
    H, mu, c_ops, rho0, meta = build_spin_resolved_projected_model(params)
    solver = LiouvilleSpectroscopySolver(solver_params)
    solver.feed_model(
        H,
        mu,
        c_ops_raw=c_ops,
        initial_density_matrix=rho0,
        density_matrix_basis="site",
    )
    return solver, meta



## Solver set-up


## Create one solver instance

This cell builds the active model from `model_params`. The printed values are
quick sanity checks: Hilbert dimension, number of momentum points, derived order
parameters, and the Hamiltonian shape in the eigenbasis.


In [5]:
#model_params["T"] = 15

solver, meta = make_solver(model_params)
print("Hilbert dimension:", meta["dim"])
print("N_k:", len(meta["k_array"]))
print("delta:", meta["delta"])
print("Hamiltonien dimensions:", solver.H_eigen.shape)

--- Model loading ---
Model transformed to the eigenbasis.
Liouville backend ready: dense.
Hilbert dimension: 9
N_k: 1
delta: 0.007071067811865476
Hamiltonien dimensions: (1, 9, 9)


### Feynmann Diagram with NQ protocol

## Pathways and spectroscopy protocol

`SolverV9` uses explicit pathways. Here the phase-discrimination string `"-++"`
asks UFSS for the third-order one-quantum pathway family corresponding to the
selected pulse phases.

Key choices:

- `maximum_manifold=1`: pathways are restricted to the singly excited manifold for this 1Q example;
- `component="chi3_1Q"`: label used later in `result.components`;
- `order=1`: select one-quantum coherence;
- `nq_interval=1`: the first propagation interval is Fourier transformed as the 1Q axis;
- `detection_interval=3`: the third interval is the emission/detection axis;
- `n_interactions=3`: third-order response.

For higher order, change the phase string, `n_interactions`, `order`,
`nq_interval`, and `maximum_manifold` consistently.


In [6]:
pathways = solver.generate_pathways_with_ufss(
    "-++",
    maximum_manifold=1,
    component="chi3_1Q",
)

protocol = standard_nq_protocol(
    order=1,
    nq_interval=1,
    detection_interval=3,
    n_interactions=3,
    nq_axis="omega_1Q",
    detection_axis="omega_emit",
)

[(pathway.name, pathway.interactions, pathway.coherence_orders) for pathway in pathways]


[('R1', ('Bu', 'Ku', 'Bd'), (-1, 0, 1)),
 ('R2', ('Bu', 'Bd', 'Ku'), (-1, 0, 1))]

### frequency grid

## Frequency grid and detection phase

The first axis is the NQ axis named by `nq_axis`; the second axis is the
detection axis named by `detection_axis`. The names in `axes={...}` must exactly
match the names in the protocol.

This example defines both rephasing and unrephasing quadrants, but the
calculation below uses `omega1_rephasing` with the `"-++"` pathways.


In [7]:
N_w = 150
omega1_rephasing = np.linspace(-1.1, -0.8, N_w)
omega1_unrephasing = np.linspace(0.8, 1.1, N_w)
omega3 = np.linspace(0.8, 1.1, N_w)
tau2 = 3
DETECTION_PHASE = np.pi / 2

print("rephasing omega1:", omega1_rephasing[0], omega1_rephasing[-1])
print("unrephasing omega1:", omega1_unrephasing[0], omega1_unrephasing[-1])
print("omega3:", omega3[0], omega3[-1])


rephasing omega1: -1.1 -0.8
unrephasing omega1: 0.8 1.1
omega3: 0.8 1.1


## Dynamics through `delays`

This is the central `SolverV9` change: the waiting time is supplied through
`delays`, not through a dedicated `tau2` solver argument.

For this third-order protocol, the only fixed time interval is `t2`, so each
spectrum uses:

```python
delays = {"t2": value}
```

The loop below calculates a sequence of spectra at different `t2` values and
stores each `SpectrumResult` in `result[i]`.


In [ ]:

result = np.zeros(15, dtype=object)
for i in range(1,15,1):
    delays = {
        "t2": 10*i
    }
 

    result[i] = solver.generate_NQ_spectrum(
        1,
        protocol,
        axes={"omega_1Q": omega1_rephasing, "omega_emit": omega3},
        delays=delays,
        pathways=pathways,
    )


Calculating 2 pathway spectrum/s on a 150x150 grid with protocol 'standard_1q' using parallel=threading, n_jobs=-1.
Using dense prefix-tree pathway reuse.
Calculating 2 pathway spectrum/s on a 150x150 grid with protocol 'standard_1q' using parallel=threading, n_jobs=-1.
Using dense prefix-tree pathway reuse.


## Save and Plot

In [ ]:
'''
save_pdf = True
output_directory = Path.cwd() / "Result_Test" / "tests"

plotter = SpectroscopyPlotter(detection_phase=DETECTION_PHASE)
plot_result = plotter.plot_pathways_multiorder(
    result,
    pathways="all",
    totals=["1Q"],
    view="imag",
    normalization="individual",
    axis_labels={
        "omega_1q": "1Q energy (eV)",
        "omega_emit": "Emission energy (eV)",
    },
    include_diagrams=False,
    display_diagrams=False,
    save_pdf=save_pdf,
    output_directory=output_directory if save_pdf else None,
    spectrum_pdf_name="5fs.pdf",
    show=True,
)
'''

## Plot pathway-resolved spectra

`plot_spectrum_result_contours(...)` reads the `SpectrumResult` returned by
`generate_NQ_spectrum(...)`. With `spectra="pathways"`, it plots individual
pathway matrices. With `totals="selected"`, it also plots the sum of the
selected pathways.

For component totals, use `spectra="components"` and choose names from
`result.components`, for example `"chi3_1Q"`, `"+1Q"`, `"-1Q"`, or `"1Q"`
depending on the protocol and selected pathways.


In [ ]:

plotter = SpectroscopyPlotter(detection_phase=DETECTION_PHASE)
for i in range(1,15,1):
    fig, axes = plotter.plot_spectrum_result_contours(
        result[i],
        spectra="pathways",
        names=["R1", "R2"],
        totals="selected",
        normalization="panel",
        color_map="Spectral_r",
        levels=12,
        show=True,
    )

## Optional: loop over one model parameter

A clean parameter scan should copy the base dictionary, overwrite one or more
keys, rebuild the solver for each case, and then call the same
pathway/protocol/spectrum workflow. This avoids mutating `model_params` in place
and makes every run reproducible.

The example below is disabled by default because it can be expensive. Set
`RUN_PARAMETER_SCAN = True` to execute it.


In [ ]:
RUN_PARAMETER_SCAN = False

if RUN_PARAMETER_SCAN:
    parameter_sets = [
        {**model_params, "label": "gQ_0", "g_Q": 0.0},
        {**model_params, "label": "gQ_0p005", "g_Q": 0.005},
        {**model_params, "label": "gQ_0p010", "g_Q": 0.010},
    ]

    scan_outputs = {}
    for params in parameter_sets:
        local_solver, local_meta = make_solver(params)
        local_pathways = local_solver.generate_pathways_with_ufss(
            "-++",
            maximum_manifold=1,
            component="chi3_1Q",
        )
        local_result = local_solver.generate_NQ_spectrum(
            1,
            protocol,
            axes={"omega_1Q": omega1_rephasing, "omega_emit": omega3},
            delays={"t2": float(spectrum_params["tau2"])},
            pathways=local_pathways,
            k_array=local_meta["k_array"],
            k_weights=local_meta["k_weights"],
            parallel_backend=solver_params.get("parallel_backend", "serial"),
            n_jobs=solver_params.get("n_jobs", 1),
            verbose=True,
        )
        scan_outputs[params["label"]] = {
            "params": params,
            "meta": local_meta,
            "result": local_result,
        }

    tuple(scan_outputs)
